In [200]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from datetime import datetime

# 自作モジュールの読み込み
sys.path.append(os.path.abspath('..'))
from configs.config import *
from src.runner import Runner
from src.model_LGBM import model_LGBM
from src.util import Logger, Util

In [201]:
# ロガーの設定
logger = Logger(path=DIR_LOG)

def get_run_name(model_type):
    """run名の作成
    """
    run_name = model_type
    suffix = '_' + datetime.now().strftime("%Y%m%d%H%M")
    run_name = run_name + suffix
    return run_name

# LGBM

In [202]:
# Key
df_all = Util.load_feature('Key')
# 目的変数
key_col = ['社員番号', 'category']
df_all = pd.merge(df_all, Util.load_feature('Target'), on=key_col, how='left')\
# category特徴量
key_col = 'category'
list_feaqture_name = [
    'CategoryFeature',
]
for feature_name in list_feaqture_name:
    df_feature = Util.load_feature(feature_name)
    df_all = pd.merge(df_all, df_feature, on=key_col, how='left')
# 社員特徴量
key_col = '社員番号'
list_feaqture_name = [
    'CareerFeature',
    'DxFeature',
    'HrFeature',
    'OvertimeWorkByMonthFeature',
    'PositionHistoryFeature',
    'UdemyActivityFeature',
]
for feature_name in list_feaqture_name:
    df_feature = Util.load_feature(feature_name)
    df_all = pd.merge(df_all, df_feature, on=key_col, how='left')

# train test
df_train = df_all[df_all['target'].notnull()]
df_test = df_all[df_all['target'].isnull()]

In [203]:
# run_nameの設定
run_name = get_run_name(model_type="lgbm_base")
# run_name = 'lgbm_multimodel_bikes_202410261847'
run_name

'lgbm_base_202507282001'

In [204]:
pos = sum(df_train['target'] == 1)
neg = sum(df_test['target'] != 1)

print(neg / pos)

38.13840830449827


In [ ]:
def after_predict_process(df_pred, target_col):
    """予測後に行う処理
    Args:
        df_pred(pd.DataFrame): 予測データ[key_cols, 予測値]
        target_col(str): 予測値のカラム名
    Returns:    
        df_pred(pd.DataFrame): 予測データ[key_cols, 予測値]
    """
    return df_pred

def after_split_process(tr, va):
    """データセットの分割後に行う処理
    Args:
        tr(pd.DataFrame): 訓練データ
        va(pd.DataFrame): 検証データ
    returns:
        tr(pd.DataFrame): 訓練データ
        va(pd.DataFrame): 検証データ
    """
    return tr, va

model_params_lgb = {
    #### run params
    "key_cols": KEY_COL,                    # ユニークキー
    "target_col": TARGET_COL,              # 目的変数（0 or 1）
    "remove_cols": [],
    #### model train params
    "num_boost_round": 5000,
    "early_stopping_rounds": 100,
    "verbose": -1,
    "period": 100,
    "log_level": 'error',
    "verbosity": -1,
    #### model core params (for binary classification)
    # "boosting_type": "gbdt",
    # "objective": "binary",                 
    # "metric": "auc",                       
    # "learning_rate": 0.01,
    # "scale_pos_weight": 10,
    # "lambda_l1": 0.1,
    # "lambda_l2": 0.1,
    # "feature_fraction": 0.8,
    # "bagging_fraction": 0.8,              # 追加: 通常は併用
    # "bagging_freq": 1,                    # 追加: バギングの頻度
    # "num_leaves": 31,                     # 追加: 複雑さの制御
    # "min_data_in_leaf": 20,               # 追加: 過学習抑制
    # "max_depth": -1,                      # 追加: 自由に展開
    # "random_state": 42,                   # 追加: 再現性確保
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": "auc",                         # または 'binary_logloss'
    "learning_rate": 0.05,
    "num_leaves": 6,                         # 指定に合わせて上書き
    "max_depth": 15,
    "feature_fraction": 0.8,                 # == colsample_bytree
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 3.5,
    "lambda_l2": 1.5,
    "min_data_in_leaf": 50,
    "colsample_bytree": 0.9,                 # LightGBM拡張用（feature_fractionとの併用に注意）
    "colsample_bynode": 0.6,
    "random_state": 42,
    "device": "cpu",
}

run_setting = {
    'calc_shap': False,     # shap値を計算するか否か
    "tune_params": True,           # パラメータチューニング、lgb_hopt,xgb_hopt,nn_hopt,False
    "after_predict_process": after_predict_process,
    'after_split_process': after_split_process,  
}
cv_setting = {
    "target_col": TARGET_COL,
    "group_col": "社員番号",  # グループ化するカラム
    "n_splits": 4,  # 分割数,
    "shuffle": True,  # シャッフルするか否か
    "random_state": 42,  # ランダムシード
}

In [218]:
import importlib
from src import runner
from src import model_LGBM
from src import model
from configs import config

# runnerモジュールをリロード
importlib.reload(runner)
importlib.reload(model_LGBM)
importlib.reload(model)
importlib.reload(config)

# Runnerクラスを再インポート
from src.model import Model
from src.runner import Runner
from src.model_LGBM import model_LGBM
from configs.config import *

In [219]:
memo = "LightGBM discussionを参考に修正"
ml_runner = Runner(
    run_name,
    model_LGBM,
    model_params_lgb,
    df_train,
    df_test,
    run_setting,
    cv_setting,
    logger,
    memo,
)

In [220]:
# ml_runner.tune_params(30)

In [221]:
ml_runner.run_train_cv()

[2025-07-28 20:02:52] - lgbm_base_202507282001 - start training cv
[2025-07-28 20:02:52] - lgbm_base_202507282001 fold 0 - start training


Training until validation scores don't improve for 100 rounds
[100]	train's auc: 0.861867	eval's auc: 0.658667
[200]	train's auc: 0.901498	eval's auc: 0.660867
[300]	train's auc: 0.921353	eval's auc: 0.66281
Early stopping, best iteration is:
[265]	train's auc: 0.914283	eval's auc: 0.66947


[2025-07-28 20:02:55] - lgbm_base_202507282001 fold 0 - end training
[2025-07-28 20:02:55] - lgbm_base_202507282001 fold 1 - start training


Training until validation scores don't improve for 100 rounds
[100]	train's auc: 0.860869	eval's auc: 0.642871
Early stopping, best iteration is:
[63]	train's auc: 0.834615	eval's auc: 0.659243


[2025-07-28 20:02:57] - lgbm_base_202507282001 fold 1 - end training
[2025-07-28 20:02:57] - lgbm_base_202507282001 fold 2 - start training


Training until validation scores don't improve for 100 rounds
[100]	train's auc: 0.863152	eval's auc: 0.630105
Early stopping, best iteration is:
[58]	train's auc: 0.829789	eval's auc: 0.643704


[2025-07-28 20:02:59] - lgbm_base_202507282001 fold 2 - end training
[2025-07-28 20:02:59] - lgbm_base_202507282001 fold 3 - start training


Training until validation scores don't improve for 100 rounds
[100]	train's auc: 0.862214	eval's auc: 0.651635
[200]	train's auc: 0.90002	eval's auc: 0.650145
Early stopping, best iteration is:
[111]	train's auc: 0.866447	eval's auc: 0.654642


[2025-07-28 20:03:01] - lgbm_base_202507282001 fold 3 - end training
[2025-07-28 20:03:01] - lgbm_base_202507282001 - end training cv


In [222]:
ml_runner.run_metric_cv()

[2025-07-28 20:03:01] - lgbm_base_202507282001 - start metric cv
100%|██████████| 4/4 [00:01<00:00,  3.37it/s]
memo: LightGBM discussionを参考に修正
run_name:lgbm_base_202507282001	score_mean:0.6567645437940308	score0:0.6694701249785995	score1:0.6592427292085995	score2:0.6437036663472692	score3:0.6546416546416547
mean: 0.6567645437940308, std: 0.009255624968353696
[2025-07-28 20:03:02] - mean: 0.6567645437940308, std: 0.009255624968353696
[2025-07-28 20:03:02] - output predict : g:\マイドライブ\competitions\atma_udemy\models\lgbm_base_202507282001\va_pred.pkl
[2025-07-28 20:03:02] - lgbm_base_202507282001 - end metric cv


In [223]:
ml_runner.run_predict_cv()

[2025-07-28 20:03:02] - lgbm_base_202507282001 - start prediction cv
100%|██████████| 4/4 [00:00<00:00, 16.50it/s]
[2025-07-28 20:03:03] - output predict : g:\マイドライブ\competitions\atma_udemy\models\lgbm_base_202507282001\te_pred.pkl
[2025-07-28 20:03:03] - lgbm_base_202507282001 - end prediction cv


In [224]:
ml_runner.plot_feature_importance_cv()

[2025-07-28 20:03:03] - lgbm_base_202507282001 - start plot feature importance cv
[2025-07-28 20:03:15] - lgbm_base_202507282001 - end plot feature importance cv


# Submissionの作成

In [225]:
runner = ml_runner

In [226]:
df_te_pred = pd.read_pickle(os.path.join(runner.out_dir_name, "te_pred.pkl"))
df_prep_test = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_test.pkl"))
df_pred = pd.merge(df_prep_test, df_te_pred, on=["社員番号", "category"], how="left")
df_submit = df_pred[['target']]

In [227]:
path_submit = os.path.join(DIR_SUBMISSIONS, f"{runner.run_name}_submition.csv")
df_submit.to_csv(path_submit, header=True, index=False)
print(path_submit)
pd.read_csv(path_submit)

g:\マイドライブ\competitions\atma_udemy\data\submission\lgbm_base_202507282001_submition.csv


,target
0,0.270281
1,0.290391
2,0.295309
3,0.270281
4,0.269781
...,...
11017,0.401157
11018,0.416813
11019,0.405302
11020,0.420980


In [228]:
df_submit['target'].describe()

count    11022.000000
mean         0.230432
std          0.111291
min          0.026638
25%          0.139258
50%          0.215830
75%          0.304649
max          0.585865
Name: target, dtype: float64